# Vigil — Colab Integration Tests

> **Runtime → Change runtime type → T4 GPU** before running anything.

Tests: OOM capture | NaN gradient | Gradient explosion | Dataloader bottleneck | Overhead benchmark

In [ ]:
import torch
assert torch.cuda.is_available(), 'Switch runtime to GPU first!'
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Replace with your GitHub URL
!git clone https://github.com/YOUR_USERNAME/Vigil.git
!pip install pynvml --quiet
!pip install -e Vigil/sdk --quiet

In [ ]:
import sys
sys.path.insert(0, 'Vigil')
import vigil, torch, torch.nn as nn
print('vigil', vigil.__version__)

## Test 1 — CUDA OOM

In [ ]:
@vigil.watch(project='colab-oom-test')
def train_oom():
    model = nn.Linear(1000, 1000).cuda()
    vigil.watch_model(model)
    for step in range(999):
        x = torch.randn(50_000, 1_000).cuda()
        loss = model(x).sum()
        loss.backward()
        vigil.step()

try:
    train_oom()
except torch.cuda.OutOfMemoryError:
    pass  # Expected -- Vigil should have printed the diagnosis above

## Test 2 — NaN Gradient (lr=1e8)

In [ ]:
@vigil.watch(project='colab-nan-test')
def train_nan():
    model = nn.Linear(10, 1).cuda()
    optimizer = torch.optim.SGD(model.parameters(), lr=1e8)
    vigil.watch_model(model)
    for step in range(50):
        x = torch.randn(32, 10).cuda()
        loss = model(x).sum()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        vigil.step()

train_nan()

## Test 3 — Dataloader Bottleneck (sleep=0.2s, num_workers=0)

In [ ]:
import time

@vigil.watch(project='colab-dl-test')
def train_slow_dl():
    model = nn.Linear(10, 1).cuda()

    class SlowDataset(torch.utils.data.Dataset):
        def __len__(self): return 20
        def __getitem__(self, i):
            time.sleep(0.2)
            return torch.randn(10), torch.randn(1)

    raw_loader = torch.utils.data.DataLoader(
        SlowDataset(), batch_size=4, num_workers=0)
    loader = vigil.watch_dataloader(raw_loader)

    for x, y in loader:
        x = x.cuda()
        loss = model(x).sum()
        loss.backward()
        vigil.step()

train_slow_dl()

## Test 4 — Overhead Benchmark (target < 0.5%)

In [ ]:
import time

STEPS, WARMUP = 200, 10

def run_steps(model, optimizer):
    times = []
    for _ in range(STEPS):
        t0 = time.perf_counter()
        x = torch.randn(64, 512, device='cuda')
        loss = model(x).sum()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return times

def build():
    m = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 256)).cuda()
    return m, torch.optim.Adam(m.parameters())

# Baseline
model_b, opt_b = build()
baseline_times = run_steps(model_b, opt_b)

# With Vigil
vigil_times = []

@vigil.watch(project='overhead-bench')
def bench_vigil():
    from vigil.emitter import Emitter, set_emitter
    s = vigil.current_session()
    s._emitter.shutdown(wait=False)
    s._emitter = Emitter(on_events=lambda evts: None)  # silent
    set_emitter(s._emitter)
    model_v, opt_v = build()
    vigil.watch_model(model_v)
    vigil_times.extend(run_steps(model_v, opt_v))

bench_vigil()

base  = sum(baseline_times[WARMUP:]) / len(baseline_times[WARMUP:])
vigil_mean = sum(vigil_times[WARMUP:]) / len(vigil_times[WARMUP:])
overhead = (vigil_mean - base) / base * 100

print(f'Baseline:   {base*1000:.3f} ms/step')
print(f'With Vigil: {vigil_mean*1000:.3f} ms/step')
print(f'Overhead:   {overhead:.3f}%')
assert overhead < 0.5, f'FAIL: overhead {overhead:.3f}% exceeds 0.5% limit'
print('PASS')